# Pipeline Machine Learning End-to-End dengan TFX

Notebook ini mengimplementasikan machine learning pipeline menggunakan TensorFlow Extended (TFX) dengan orchestrator `InteractiveContext`. Pipeline mencakup validasi data otomatis, preprocessing, hyperparameter tuning, training, evaluasi berbasis validation blessing, hingga model deployment.

## Inisialisasi dan Konfigurasi Pipeline

In [1]:
import os
import tensorflow as tf
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext

2026-09-12 19:03:52.950005: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
USERNAME = 'erlanggajuni45'

PIPELINE_NAME = f"{USERNAME}-pipeline"
SCHEMA_PIPELINE_NAME = PIPELINE_NAME
PIPELINE_ROOT = os.path.join(PIPELINE_NAME, 'pipeline_root')
METADATA_PATH = os.path.join(PIPELINE_NAME, 'metadata', 'metadata.db')
DATA_ROOT = os.path.join(PIPELINE_NAME, 'data')
SERVING_MODEL_DIR = os.path.join(PIPELINE_NAME, 'serving_model')

In [3]:
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(SERVING_MODEL_DIR, exist_ok=True)

Inisialiasasi InteractiveContext dengan SQLite metadata

In [4]:
context = InteractiveContext(
    pipeline_name=PIPELINE_NAME,
    pipeline_root=PIPELINE_ROOT,
    metadata_connection_config=None
)

## 1. Data Ingestion (CsvExampleGen)
Komponen `CsvExampleGen` membaca file CSV dari direktori sumber, melakukan konversi data ke format `tf.train.example` berekstensi TFRecord, serta membagi data secara otomatis menjadi partisi `train` dan `eval`

In [5]:
from tfx.components import CsvExampleGen

example_gen = CsvExampleGen(input_base=DATA_ROOT)
context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 35
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

## 2. Ringkasan Statistik Data (StatisticsGen)
Komponen `StatisticsGen` menghitung ringkasan statistik deskriptif dari data latih dan evaluasi menggunakan TensorFlow Data Validation (TFDV). Statistik ini memetakan sebaran nilai numerik, kuantil, *missing values*, serta frekuensi kategori.

In [6]:
from tfx.components import StatisticsGen

statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])
context.run(statistics_gen)

ExecutionResult(
    component_id: StatisticsGen
    execution_id: 36
    outputs:
        statistics: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

Menampilkan visualisasi distribusi data

In [7]:
context.show(statistics_gen.outputs['statistics'])

## 3. Inferensi Skema Data (SchemaGen)
Komponen `SchemaGen` menganalisis output statistik dari `StatisticsGen` untuk menetapkan kontrak skema data (*data schema*), mencakup tipe data tiap kolom, nilai domain kategori, dan batasan nilai yang diharapkan.

In [8]:
from tfx.components import SchemaGen

schema_gen = SchemaGen(statistics_gen.outputs['statistics'], infer_feature_shape=True)
context.run(schema_gen)

ExecutionResult(
    component_id: SchemaGen
    execution_id: 37
    outputs:
        schema: OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

menampilkan skema yang berhasil diinferensi

In [9]:
context.show(schema_gen.outputs['schema'])

,Type,Presence,Valency,Domain
Feature name,,,,
'bathrooms',FLOAT,required,,-
'bedrooms',FLOAT,required,,-
'city',STRING,required,,'city'
'condition',INT,required,,-
'floors',FLOAT,required,,-
'price',FLOAT,required,,-
'sqft_above',INT,required,,-
'sqft_basement',INT,required,,-
'sqft_living',INT,required,,-


,Values
Domain,
'city',"'Algona', 'Auburn', 'Bellevue', 'Black Diamond', 'Bothell', 'Burien', 'Carnation', 'Clyde Hill', 'Covington', 'Des Moines', 'Duvall', 'Enumclaw', 'Fall City', 'Federal Way', 'Inglewood-Finn Hill', 'Issaquah', 'Kenmore', 'Kent', 'Kirkland', 'Lake Forest Park', 'Maple Valley', 'Medina', 'Mercer Island', 'Milton', 'Newcastle', 'Normandy Park', 'North Bend', 'Pacific', 'Preston', 'Ravensdale', 'Redmond', 'Renton', 'Sammamish', 'SeaTac', 'Seattle', 'Shoreline', 'Skykomish', 'Snoqualmie', 'Snoqualmie Pass', 'Tukwila', 'Vashon', 'Woodinville', 'Yarrow Point', 'Beaux Arts Village'"
'statezip',"'WA 98001', 'WA 98002', 'WA 98003', 'WA 98004', 'WA 98005', 'WA 98006', 'WA 98007', 'WA 98008', 'WA 98010', 'WA 98011', 'WA 98014', 'WA 98019', 'WA 98022', 'WA 98023', 'WA 98024', 'WA 98027', 'WA 98028', 'WA 98029', 'WA 98030', 'WA 98031', 'WA 98032', 'WA 98033', 'WA 98034', 'WA 98038', 'WA 98039', 'WA 98040', 'WA 98042', 'WA 98045', 'WA 98047', 'WA 98050', 'WA 98051', 'WA 98052', 'WA 98053', 'WA 98055', 'WA 98056', 'WA 98057', 'WA 98058', 'WA 98059', 'WA 98065', 'WA 98068', 'WA 98070', 'WA 98072', 'WA 98074', 'WA 98075', 'WA 98077', 'WA 98092', 'WA 98102', 'WA 98103', 'WA 98105', 'WA 98106', 'WA 98107', 'WA 98108', 'WA 98109', 'WA 98112', 'WA 98115', 'WA 98116', 'WA 98117', 'WA 98118', 'WA 98119', 'WA 98122', 'WA 98125', 'WA 98126', 'WA 98133', 'WA 98136', 'WA 98144', 'WA 98146', 'WA 98148', 'WA 98155', 'WA 98166', 'WA 98168', 'WA 98177', 'WA 98178', 'WA 98188', 'WA 98198', 'WA 98199', 'WA 98288', 'WA 98354'"


## 4. Validasi Anomali Dataa (ExampleValidator)
Komponen `ExampleValidator` memverifikasi statistik data terhadap skema yang telah terbentuk. Komponen ini mendeteksi ada atau tidaknya *anomalies*, *data drift*, atau ketidaksesuaian tipe data pada dataset.

In [10]:
from tfx.components import ExampleValidator

example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema'],
)
context.run(example_validator)

ExecutionResult(
    component_id: ExampleValidator
    execution_id: 38
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

menampilkan laporan anomali

In [11]:
context.show(example_validator.outputs['anomalies'])

## 5. Data Preprocessing (Transform)
Komponen `Transform` melakukan rekayasa fitur (*feature engineering*) secara konsisten pada tahap pelatihan maupun inferensi produksi (mencegah *training-serving skew*). Fitur numerik distandarisasi menggunakan z-score (`tft.scale_to_z_score`), sedangkan fitur kategorikal diubah menjadi indeks numerik menggunakan representasi kosakata (`tft.compute_and_apply_vocabulary`)

menuliskan modul transform ke file python

In [12]:
%%writefile modules/transform.py
import tensorflow as tf
import tensorflow_transform as tft

LABEL_KEY = 'price'

NUMERICAL_FEATURES = [
    'bathrooms',
    'bedrooms',
    'condition',
    'floors',
    'sqft_above',
    'sqft_basement',
    'sqft_living',
    'sqft_lot',
    'view',
    'waterfront',
    'yr_built',
    'yr_renovated'
]

CATEGORICAL_FEATURES = [
    'city',
    'statezip'
]

def transformed_name(key: str) -> str:
    """Mengubah nama fitur asli menjadi nama fitur hasil transformasi"""
    return f"{key}_xf"

def preprocessing_fn(inputs):
    """Fungsi transformasi fitur menggunakan TensorFlow Transform"""
    outputs = {}

    # Standarisasi fitur numerik dengan z-score
    for feature in NUMERICAL_FEATURES:
        outputs[transformed_name(feature)] = tft.scale_to_z_score(
            tf.cast(inputs[feature], tf.float32)
        )

    # Encoding fitur kategorikal ke vocabulary index
    for feature in CATEGORICAL_FEATURES:
        outputs[transformed_name(feature)] = tft.compute_and_apply_vocabulary(
            tf.cast(inputs[feature], tf.string)
        )

    # Label target dipastikan bertipe float32
    outputs[transformed_name(LABEL_KEY)] = tf.cast(inputs[LABEL_KEY], tf.float32)

    return outputs

Overwriting modules/transform.py


Menjalankan komponen Transform

In [13]:
from tfx.components import Transform

TRANSFORM_MODULE_FILE = 'modules/transform.py'

transform = Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file=TRANSFORM_MODULE_FILE,
)
context.run(transform)

running bdist_wheel
running build
running build_py
creating build/lib
copying transform.py -> build/lib
copying trainer.py -> build/lib
copying tuner.py -> build/lib
installing to /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmp8mgn2k6g
running install
running install_lib


/Users/erlanggajuni/anaconda3/envs/mlops-tfx-submission/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


copying build/lib/transform.py -> /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmp8mgn2k6g/.
copying build/lib/trainer.py -> /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmp8mgn2k6g/.
copying build/lib/tuner.py -> /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmp8mgn2k6g/.
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmp8mgn2k6g/./tfx_user_code_Transform-0.0+2dbfae355d5d80eb809f58d98fe990e8cbade3dfa78fd3e647b05eb9d8e38970-py3.10.egg-info
ru

INFO:tensorflow:Assets written to: erlanggajuni45-pipeline/pipeline_root/Transform/transform_graph/39/.temp_path/tftransform_tmp/38d640bc129543d1bd4b62f99748e7ab/assets


INFO:tensorflow:Assets written to: erlanggajuni45-pipeline/pipeline_root/Transform/transform_graph/39/.temp_path/tftransform_tmp/38d640bc129543d1bd4b62f99748e7ab/assets


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 39
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

## 6. Hyperparameter Tuning (Tuner)
Komponen `Tuner` melakukan optimasi hyperparameter model secara otomatis menggunakan KerasTuner (`RandomSearch`). Parameter yang dicari mencakup jumlah unit dense layer, dropout rate, dan learning rate optimizer Adam dengan metrik objektif meminimalkan Mean Absolute Error (`val_mean_absolute_error`).

menuliskan modul tuner ke file python

In [14]:
%%writefile modules/tuner.py
from typing import NamedTuple, Dict, Text, Any
import keras_tuner as kt
import tensorflow as tf
import tensorflow_transform as tft
from tfx.components.trainer.fn_args_utils import FnArgs

LABEL_KEY = 'price'

NUMERICAL_FEATURES = [
    'bathrooms', 'bedrooms', 'condition', 'floors',
    'sqft_above', 'sqft_basement', 'sqft_living', 'sqft_lot',
    'view', 'waterfront', 'yr_built', 'yr_renovated'
]

CATEGORICAL_FEATURES = ['city', 'statezip']

def transformed_name(key: str) -> str:
    return f"{key}_xf"

def _gzip_reader_fn(filenames):
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

def _input_fn(file_pattern, tf_transform_output, batch_size=32):
    transformed_feature_spec = (
        tf_transform_output.transformed_feature_spec().copy()
    )

    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transformed_feature_spec,
        reader=_gzip_reader_fn,
        num_epochs=None,
        label_key=transformed_name(LABEL_KEY)
    )
    return dataset

def model_builder(hp: kt.HyperParameters) -> tf.keras.Model:
    inputs = {}

    # Layer input numerik
    for feat in NUMERICAL_FEATURES:
        inputs[transformed_name(feat)] = tf.keras.layers.Input(
            shape=(1,), name=transformed_name(feat), dtype=tf.float32
        )

    # Layer input kategorikal
    for feat in CATEGORICAL_FEATURES:
        inputs[transformed_name(feat)] = tf.keras.layers.Input(
            shape=(1,), name=transformed_name(feat), dtype=tf.int64
        )

    # Konversi categorical index ke embedding representation
    cat_embeddings = []
    for feat in CATEGORICAL_FEATURES:
        embed = tf.keras.layers.Embedding(
            input_dim=150, output_dim=8
        )(inputs[transformed_name(feat)])
        cat_embeddings.append(tf.keras.layers.Flatten()(embed))

    num_layers = [inputs[transformed_name(f)] for f in NUMERICAL_FEATURES]
    all_features = tf.keras.layers.concatenate(num_layers + cat_embeddings)

    # Tuning jumlah unit layer dense
    units_1 = hp.Int('units_1', min_value=32, max_value=128, step=32, default=64)
    x = tf.keras.layers.Dense(units_1, activation='relu')(all_features)

    # Tuning dropout
    dropout_rate = hp.Float('dropout_rate', min_value=0.1, max_value=0.4, step=0.1, default=0.2)
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    units_2 = hp.Int('units_2', min_value=16, max_value=64, step=16, default=32)
    x = tf.keras.layers.Dense(units_2, activation='relu')(x)

    output = tf.keras.layers.Dense(1, activation='linear')(x)

    model = tf.keras.Model(inputs=inputs, outputs=output)

    # Tuning learning rate
    lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 5e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='mean_squared_error',
        metrics=[tf.keras.metrics.MeanAbsoluteError(name='mean_absolute_error')]
    )
    return model

TunerFnResult = NamedTuple('TunerFnResult', [
    ('tuner', kt.engine.base_tuner.BaseTuner),
    ('fit_kwargs', Dict[Text, Any]),
])

def tuner_fn(fn_args: FnArgs) -> TunerFnResult:
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)

    train_set = _input_fn(fn_args.train_files, tf_transform_output, batch_size=64)
    eval_set = _input_fn(fn_args.eval_files, tf_transform_output, batch_size=64)

    tuner = kt.RandomSearch(
        hypermodel=model_builder,
        objective=kt.Objective('val_mean_absolute_error', direction='min'),
        max_trials=3,
        executions_per_trial=1,
        directory=fn_args.working_dir,
        project_name='house_price_tuning'
    )

    return TunerFnResult(
        tuner=tuner,
        fit_kwargs={
            'x': train_set,
            'validation_data': eval_set,
            'steps_per_epoch': fn_args.train_steps,
            'validation_steps': fn_args.eval_steps,
            'epochs': 5
        }
    )

Overwriting modules/tuner.py


Menjalankan komponen Tuner

In [15]:
from tfx.components import Tuner
from tfx.proto import trainer_pb2

TUNER_MODULE_FILE = 'modules/tuner.py'

tuner = Tuner (
    module_file=TUNER_MODULE_FILE,
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=40),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=20),
)
context.run(tuner)

Trial 3 Complete [00h 00m 06s]
val_mean_absolute_error: 538907.4375

Best val_mean_absolute_error So Far: 535016.375
Total elapsed time: 00h 00m 21s
Results summary
Results in erlanggajuni45-pipeline/pipeline_root/.temp/40/house_price_tuning
Showing 10 best trials
Objective(name="val_mean_absolute_error", direction="min")

Trial 0 summary
Hyperparameters:
units_1: 32
dropout_rate: 0.1
units_2: 48
learning_rate: 0.0005
Score: 535016.375

Trial 1 summary
Hyperparameters:
units_1: 64
dropout_rate: 0.2
units_2: 32
learning_rate: 0.001
Score: 538049.125

Trial 2 summary
Hyperparameters:
units_1: 96
dropout_rate: 0.2
units_2: 64
learning_rate: 0.0005
Score: 538907.4375


ExecutionResult(
    component_id: Tuner
    execution_id: 40
    outputs:
        best_hyperparameters: OutputChannel(artifact_type=HyperParameters, producer_component_id=Tuner, output_key=best_hyperparameters, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        tuner_results: OutputChannel(artifact_type=TunerResults, producer_component_id=Tuner, output_key=tuner_results, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

## 7. Pelatihan Model (Trainer)
Komponen `Trainer` melatih model TensorFlow Keras secara penuh menggunakan konfigurasi arsitektur dan parameter terbaik dari komponen `Tuner`.  Model akhir diekspor dalam format SavedModel lengkap dengan *serving signature* (`serve_tf_examples_fn`) agar dapat memproses data mentah langsung saat inferensi dan evaluasi.

menulis modul trainer di file python

In [19]:
%%writefile modules/trainer.py
import tensorflow as tf
import tensorflow_transform as tft
import tf_keras as keras
from tfx.components.trainer.fn_args_utils import FnArgs

LABEL_KEY = 'price'

NUMERICAL_FEATURES = [
    'bathrooms', 'bedrooms', 'condition', 'floors',
    'sqft_above', 'sqft_basement', 'sqft_living', 'sqft_lot',
    'view', 'waterfront', 'yr_built', 'yr_renovated'
]

CATEGORICAL_FEATURES = ['city', 'statezip']

def transformed_name(key: str) -> str:
    return f"{key}_xf"

def _gzip_reader_fn(filenames):
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

def _input_fn(file_pattern, tf_transform_output, batch_size=32):
    transformed_feature_spec = (
        tf_transform_output.transformed_feature_spec().copy()
    )

    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transformed_feature_spec,
        reader=_gzip_reader_fn,
        num_epochs=None,
        label_key=transformed_name(LABEL_KEY)
    )
    return dataset

def _build_keras_model(hp_dict: dict) -> keras.Model:
    inputs = {}

    for feat in NUMERICAL_FEATURES:
        inputs[transformed_name(feat)] = keras.layers.Input(
            shape=(1,), name=transformed_name(feat), dtype=tf.float32
        )

    for feat in CATEGORICAL_FEATURES:
        inputs[transformed_name(feat)] = keras.layers.Input(
            shape=(1,), name=transformed_name(feat), dtype=tf.int64
        )

    cat_embeddings = []
    for feat in CATEGORICAL_FEATURES:
        feat_input = inputs[transformed_name(feat)]
        
        # Mengubah indeks OOV (-1) menjadi 0 dan membatasi rentang ke [0, 149]
        cleaned_input = keras.layers.Lambda(
            lambda x: tf.clip_by_value(x, tf.cast(0, x.dtype), tf.cast(149, x.dtype))
        )(feat_input)
        
        embed = keras.layers.Embedding(
            input_dim=150, output_dim=8
        )(cleaned_input)
        cat_embeddings.append(keras.layers.Flatten()(embed))

    num_layers = [inputs[transformed_name(f)] for f in NUMERICAL_FEATURES]
    all_features = keras.layers.concatenate(num_layers + cat_embeddings)

    units_1 = hp_dict.get('units_1', 64)
    dropout_rate = hp_dict.get('dropout_rate', 0.2)
    units_2 = hp_dict.get('units_2', 32)
    learning_rate = hp_dict.get('learning_rate', 0.001)

    x = keras.layers.Dense(units_1, activation='relu')(all_features)
    x = keras.layers.Dropout(dropout_rate)(x)
    x = keras.layers.Dense(units_2, activation='relu')(x)
    output = keras.layers.Dense(1, activation='linear')(x)

    model = keras.Model(inputs=inputs, outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mean_squared_error',
        metrics=[keras.metrics.MeanAbsoluteError(name='mean_absolute_error')]
    )
    return model

def _get_serve_tf_examples_fn(model, tf_transform_output):
    model.tft_layer = tf_transform_output.transform_features_layer()

    @tf.function(
        input_signature=[
            tf.TensorSpec(shape=[None], dtype=tf.string, name='examples')
        ]
    )
    def serve_tf_examples_fn(serialized_tf_examples):
        feature_spec = tf_transform_output.raw_feature_spec()
        feature_spec.pop(LABEL_KEY, None)
        parsed_features = tf.io.parse_example(serialized_tf_examples, feature_spec)
        transformed_features = model.tft_layer(parsed_features)
        return model(transformed_features)

    return serve_tf_examples_fn

def run_fn(fn_args: FnArgs):
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)

    train_dataset = _input_fn(fn_args.train_files, tf_transform_output, batch_size=64)
    eval_dataset = _input_fn(fn_args.eval_files, tf_transform_output, batch_size=64)

    hp_dict = fn_args.hyperparameters.get('values', {}) if fn_args.hyperparameters else {}

    model = _build_keras_model(hp_dict)

    model.fit(
        train_dataset,
        steps_per_epoch=fn_args.train_steps,
        validation_data=eval_dataset,
        validation_steps=fn_args.eval_steps,
        epochs=15
    )

    signatures = {
        'serving_default': _get_serve_tf_examples_fn(model, tf_transform_output),
    }

    model.save(
        fn_args.serving_model_dir,
        save_format='tf',
        signatures=signatures
    )

Overwriting modules/trainer.py


menjalankan komponen Trainer

In [20]:
from tfx.components import Trainer
from tfx.proto import trainer_pb2

TRAINER_MODULE_FILE = "modules/trainer.py"

trainer = Trainer(
    module_file=TRAINER_MODULE_FILE,
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    hyperparameters=tuner.outputs["best_hyperparameters"],
    train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=100),
    eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=50),
)
context.run(trainer)

running bdist_wheel
running build
running build_py
creating build/lib
copying transform.py -> build/lib
copying trainer.py -> build/lib
copying tuner.py -> build/lib
installing to /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpx4n651pd
running install
running install_lib


/Users/erlanggajuni/anaconda3/envs/mlops-tfx-submission/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


copying build/lib/transform.py -> /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpx4n651pd/.
copying build/lib/trainer.py -> /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpx4n651pd/.
copying build/lib/tuner.py -> /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpx4n651pd/.
running install_egg_info
running egg_info
creating tfx_user_code_Trainer.egg-info
writing tfx_user_code_Trainer.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Trainer.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Trainer.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
Copying tfx_user_code_Trainer.egg-info to /var/folders/dx/x3n1l5g90nl_ntfwj__7qh7c0000gn/T/tmpx4n651pd/./tfx_user_code_Trainer-0.0+dc51a6f49355c98c73d52a8ca491a7af3f981dc35160aa3ebc19bd7fb1b3cf29-py3.10.egg-info
running install_scri

Processing ./erlanggajuni45-pipeline/pipeline_root/_wheels/tfx_user_code_trainer-0.0+dc51a6f49355c98c73d52a8ca491a7af3f981dc35160aa3ebc19bd7fb1b3cf29-py3-none-any.whl
Epoch 1/15
100/100 [==============================] - 6s 23ms/step - loss: 657760387072.0000 - mean_absolute_error: 552196.3750 - val_loss: 534596812800.0000 - val_mean_absolute_error: 553314.4375
Epoch 2/15
100/100 [==============================] - 0s 5ms/step - loss: 652097290240.0000 - mean_absolute_error: 550811.6250 - val_loss: 533605842944.0000 - val_mean_absolute_error: 552354.0625
Epoch 3/15
100/100 [==============================] - 0s 4ms/step - loss: 763085651968.0000 - mean_absolute_error: 555933.7500 - val_loss: 587668193280.0000 - val_mean_absolute_error: 556448.5000
Epoch 4/15
100/100 [==============================] - 0s 4ms/step - loss: 546351710208.0000 - mean_absolute_error: 545608.8125 - val_loss: 530320785408.0000 - val_mean_absolute_error: 550491.1250
Epoch 5/15
100/100 [============================

INFO:tensorflow:struct2tensor is not available.


...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: erlanggajuni45-pipeline/pipeline_root/Trainer/model/43/Format-Serving/assets


INFO:tensorflow:Assets written to: erlanggajuni45-pipeline/pipeline_root/Trainer/model/43/Format-Serving/assets


ExecutionResult(
    component_id: Trainer
    execution_id: 43
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

## 8. Model Resolver (Resolver)
Komponen `Resolver` menentukan model baseline terdahulu yang disetujui (*blessed*) dari ML Metadata store menggunakan strategi `LatestBlessedModelStrategy`. Model baseline ini akan dibandingkan dengan model kandidat baru pada tahap evaluasi.

In [21]:
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
).with_id('latest_blessed_model_resolver')

context.run(model_resolver)

ExecutionResult(
    component_id: latest_blessed_model_resolver
    execution_id: 44
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=latest_blessed_model_resolver, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=latest_blessed_model_resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

## 9. Evaluasi & Validasi Model (Evaluator)
Komponen `Evaluator` menguji performa model terhadap data evaluasi menggunakan TensorFlow Model Analysis (TFMA). Komponen ini menetapkan ambang batas metrik (*metric threshold*) untuk memastikan model yang diekspor memenuhi standar kualitas produksi serta menghasilkan artefak `blessing`.

In [22]:
import tensorflow_model_analysis as tfma
from tfx.components import Evaluator

eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key='price')],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(class_name='MeanAbsoluteError'),
                tfma.MetricConfig(
                    class_name='MeanSquaredError',
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            upper_bound={'value': 1e14}
                        )
                    ),
                ),
            ]
        )
    ],
)

evaluator = Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config,
)
context.run(evaluator)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 45
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

### Menampilkan hasil evaluasi

In [29]:
import tensorflow_model_analysis as tfma

eval_result = evaluator.outputs['evaluation'].get()[0].uri
tfma_result = tfma.load_eval_result(eval_result)

tfma.view.render_slicing_metrics(tfma_result)

SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'Overall', 'metrics':…

In [32]:
import pandas as pd

# Menampilkan metrik Evaluator (TFMA) ke dalam bentuk tabel
records = []
for slice_key, sub_dict in tfma_result.slicing_metrics:
  metrics_data = sub_dict.get('', {}).get('', {})
  for metric_name, val_dict in metrics_data.items():
    val = (
        val_dict.get('doubleValue')
        if isinstance(val_dict, dict)
        else val_dict
    )
    records.append({'Metrik Evaluator (TFMA)': metric_name, 'Nilai': val})

df_eval = pd.DataFrame(records)
display(df_eval)

,Metrik Evaluator (TFMA),Nilai
0,mean_absolute_error,4.777485e+05
1,mean_squared_error,4.544436e+11
2,mean_absolute_error_diff,3.510680e+05
3,mean_squared_error_diff,3.043734e+11


## 10. Deployment Model (Pusher)
Komponen `Pusher` memeriksa status validasi (*blessing*) dari komponen `Evaluator`. Jika model dinyatakan valid (`BLESSED`), komponen ini akan menyalin model ke direktori tujuan serving (`SERVING_MODEL_DIR`) agar siap di-load oleh TensorFlow Serving.

In [23]:
from tfx.components import Pusher
from tfx.proto import pusher_pb2

pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    ),
)

context.run(pusher)

ExecutionResult(
    component_id: Pusher
    execution_id: 46
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))